## Baysian Network

Simple notebook to experiment with BN

In [38]:
import pandas as pd

tele = pd.read_csv("./../data/raw/telemetry.csv", parse_dates=["timestamp"])
labels = pd.read_csv("./../data/processed/labels-3.csv", parse_dates=["timestamp"])

df = tele.merge(labels, on=["timestamp", "machine_id"])

df.head()

,timestamp,machine_id,spindle_temp,ambient_temp,vibration_rms,coolant_flow,feed_rate,spindle_speed,load_pct,power_kw,tool_wear,spindle_overheat
0,2025-01-01 00:00:00,M-A,73.96,24.36,0.871,1.050,0.9,3600,0.294,3.193,0.000,0
1,2025-01-01 00:05:00,M-A,53.38,22.87,0.536,1.083,1.3,800,0.329,2.149,0.000,0
2,2025-01-01 00:10:00,M-A,58.12,24.59,0.455,1.025,0.9,1200,0.369,2.457,0.001,0
3,2025-01-01 00:15:00,M-A,58.08,23.05,0.680,0.977,0.7,1200,0.479,2.702,0.001,0
4,2025-01-01 00:20:00,M-A,55.37,24.28,0.663,1.072,1.1,800,0.328,2.148,0.001,0


In [39]:
df["delta_temp"] = df["spindle_temp"] - df["ambient_temp"]

vib_hi = df["vibration_rms"].quantile(0.9)
temp_hi = df["delta_temp"].quantile(0.9)
flow_lo = df["coolant_flow"].quantile(0.1)

df["high_vibration"] = (df["vibration_rms"] > vib_hi).astype(int)
df["high_spindle_temp"] = (df["delta_temp"] > temp_hi).astype(int)
df["low_coolant_flow"] = (df["coolant_flow"] < flow_lo).astype(int)
df["spindle_overheat"] = df["spindle_overheat"].astype(int)
df.head()

,timestamp,machine_id,spindle_temp,ambient_temp,vibration_rms,coolant_flow,feed_rate,spindle_speed,load_pct,power_kw,tool_wear,spindle_overheat,delta_temp,high_vibration,high_spindle_temp,low_coolant_flow
0,2025-01-01 00:00:00,M-A,73.96,24.36,0.871,1.050,0.9,3600,0.294,3.193,0.000,0,49.60,0,0,0
1,2025-01-01 00:05:00,M-A,53.38,22.87,0.536,1.083,1.3,800,0.329,2.149,0.000,0,30.51,0,0,0
2,2025-01-01 00:10:00,M-A,58.12,24.59,0.455,1.025,0.9,1200,0.369,2.457,0.001,0,33.53,0,0,0
3,2025-01-01 00:15:00,M-A,58.08,23.05,0.680,0.977,0.7,1200,0.479,2.702,0.001,0,35.03,0,0,0
4,2025-01-01 00:20:00,M-A,55.37,24.28,0.663,1.072,1.1,800,0.328,2.148,0.001,0,31.09,0,0,0


## MLE model

In this case all data are observed.

In [40]:
from pgmpy.models import DiscreteBayesianNetwork

model1 = DiscreteBayesianNetwork(
    ebunch=[
        ("spindle_overheat", "high_vibration"),
        ("spindle_overheat", "high_spindle_temp"),
        ("spindle_overheat", "low_coolant_flow"),
    ],
)

In [41]:
from pgmpy.estimators import MaximumLikelihoodEstimator

data1 = df[["spindle_overheat", "high_vibration", "high_spindle_temp", "low_coolant_flow"]]
model1.fit(data1, estimator=MaximumLikelihoodEstimator)

model1.check_model()

INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'spindle_overheat': 'N', 'high_vibration': 'N', 'high_spindle_temp': 'N', 'low_coolant_flow': 'N'}


True

In [42]:
from pgmpy.inference import VariableElimination

infer1 = VariableElimination(model1)

q1 = infer1.query(
    variables=["spindle_overheat"],
    evidence={"high_vibration": 1, "low_coolant_flow": 1, "high_spindle_temp": 1},
)
print(q1.values)
print(q1)

[0. 1.]
+---------------------+-------------------------+
| spindle_overheat    |   phi(spindle_overheat) |
+=====================+=========================+
| spindle_overheat(0) |                  0.0000 |
+---------------------+-------------------------+
| spindle_overheat(1) |                  1.0000 |
+---------------------+-------------------------+


## EM model

In Expectation Maximization model there are hidden data, such as: `bearing_wear_high`, `clogged_filter`, `fan_fault`, `low_cooling_efficiency`.

In [43]:
from pgmpy.models import DiscreteBayesianNetwork

model2 = DiscreteBayesianNetwork(
    ebunch=[
        ("bearing_wear_high", "high_vibration"),
        ("bearing_wear_high", "spindle_overheat"),
        ("fan_fault", "high_spindle_temp"),
        ("fan_fault", "spindle_overheat"),
        ("clogged_filter", "low_coolant_flow"),
        ("clogged_filter", "spindle_overheat"),
        ("low_cooling_efficiency", "high_spindle_temp"),
        ("low_cooling_efficiency", "spindle_overheat"),
    ],
    latents={
        "bearing_wear_high", 
        "clogged_filter", 
        "fan_fault", 
        "low_cooling_efficiency"
    },
)

In [44]:
from pgmpy.estimators import ExpectationMaximization

data2 = df[["spindle_overheat", "high_vibration", "high_spindle_temp", "low_coolant_flow"]]

em_est = ExpectationMaximization(model=model2, data=data2)

cpds_em = em_est.get_parameters(
    latent_card={lv: 2 for lv in ["bearing_wear_high", "fan_fault", "clogged_filter", "low_cooling_efficiency"]},
    max_iter=1000,
    atol=1e-8,
    n_jobs=1,
    seed=42,
    show_progress=True,
)

model2.add_cpds(*cpds_em)

model2.check_model()

INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'spindle_overheat': 'N', 'high_vibration': 'N', 'high_spindle_temp': 'N', 'low_coolant_flow': 'N'}
  5%|▌         | 53/1000 [00:01<00:19, 48.41it/s]


True

In [45]:
from pgmpy.inference import VariableElimination

infer2 = VariableElimination(model2)

evidence = {
    "high_vibration": 1,
    "high_spindle_temp": 1,
    "low_coolant_flow": 1,
}

# 1) Overheat probability
q_overheat = infer2.query(
    variables=["spindle_overheat"],
    evidence=evidence,
)
print("P(SpindleOverheat | evidence):")
print(q_overheat)

# 2) Posterior over latent causes
q_causes = infer2.query(
    variables=["bearing_wear_high", "fan_fault", "clogged_filter", "low_cooling_efficiency"],
    evidence=evidence,
)
print("\nP(Causes | evidence):")
print(q_causes)

P(SpindleOverheat | evidence):
+---------------------+-------------------------+
| spindle_overheat    |   phi(spindle_overheat) |
+=====================+=========================+
| spindle_overheat(0) |                  0.0000 |
+---------------------+-------------------------+
| spindle_overheat(1) |                  1.0000 |
+---------------------+-------------------------+

P(Causes | evidence):
+----------------------+--------------+-------------------+---------------------------+--------------------------------------------------------------------------+
| bearing_wear_high    | fan_fault    | clogged_filter    | low_cooling_efficiency    |   phi(bearing_wear_high,fan_fault,clogged_filter,low_cooling_efficiency) |
+======================+==============+===================+===========================+==========================================================================+
| bearing_wear_high(0) | fan_fault(0) | clogged_filter(0) | low_cooling_efficiency(0) |                    

In [46]:
q_bearing = infer2.query(
    variables=["bearing_wear_high"],
    evidence=evidence,
)
print(q_bearing)

q_fan = infer2.query(
    variables=["fan_fault"],
    evidence=evidence,
)
print(q_fan)

q_clogged = infer2.query(
    variables=["clogged_filter"],
    evidence=evidence,
)
print(q_clogged)

q_cooling = infer2.query(
    variables=["low_cooling_efficiency"],
    evidence=evidence,
)
print(q_cooling)

+----------------------+--------------------------+
| bearing_wear_high    |   phi(bearing_wear_high) |
+======================+==========================+
| bearing_wear_high(0) |                   0.0000 |
+----------------------+--------------------------+
| bearing_wear_high(1) |                   1.0000 |
+----------------------+--------------------------+
+--------------+------------------+
| fan_fault    |   phi(fan_fault) |
+==============+==================+
| fan_fault(0) |           0.0000 |
+--------------+------------------+
| fan_fault(1) |           1.0000 |
+--------------+------------------+
+-------------------+-----------------------+
| clogged_filter    |   phi(clogged_filter) |
+===================+=======================+
| clogged_filter(0) |                0.0000 |
+-------------------+-----------------------+
| clogged_filter(1) |                1.0000 |
+-------------------+-----------------------+
+---------------------------+-------------------------------+
